<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Core landscape &mdash; who and where</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        The "who / where / when" battery: where the antibiotic-resistance families are filed, how the
        field internationalises, how big the families are, and which organisations lead it.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; landscape analyses after <span style="color: #be0f05; font-weight: 600;">Riccardo Priore</span>, Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1&ndash;2 &nbsp;&middot;&nbsp; Load the corpus, retrieve filings &amp; authorities
            <br/>Step&nbsp;3&ndash;4 &nbsp;&middot;&nbsp; Filings by international / regional authority; WO vs EP over time
            <br/>Step&nbsp;5&ndash;6 &nbsp;&middot;&nbsp; National filing trends; innovation waves by technology area
            <br/>Step&nbsp;7&ndash;8 &nbsp;&middot;&nbsp; National vs international strategy; family size &amp; global reach
            <br/>Step&nbsp;9&ndash;10 &nbsp;&middot;&nbsp; Top applicants by families; applicants by institutional sector
            <br/>Step&nbsp;11&ndash;12 &nbsp;&middot;&nbsp; Grant rate by top applicants; most influential organisations
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 2 of 4 &mdash; run the four notebooks of this module in order.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Re-running needs <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via <code>epo.tipdata</code>).
            Each notebook writes what the next one reads; notebook&nbsp;4 assembles the report.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

## Step 1 — Load the shared corpus

We read `dataset.xlsx` from notebook 1 and take its `docdb_family_id` list. Every analysis in this module starts exactly here, so the whole course rests on one corpus.

In [ ]:
import pandas as pd
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS211_PAT_PUBLN
import report_kit

patstat = PatstatClient(env='PROD')
db = patstat.orm()

OUT = "2_core_landscape_analyses_output"
import os; os.makedirs(OUT, exist_ok=True)

dataset = pd.read_excel("1_dataset_and_search_strategy_output/dataset.xlsx")
family_ids = dataset['docdb_family_id'].unique().tolist()
print(f"{len(family_ids):,} families loaded from the shared corpus.")

## Step 2 — Retrieve filings and their authorities

For each family we pull its applications and the **publication authority** (`publn_auth`) — the office where it was filed — together with the application's filing year. We keep one row per distinct *application*, so a family filed at several offices contributes to each. (Note: this counts **filings**, i.e. applications, not families — a family with two EP applications counts twice in the EP bar.)

In [ ]:
rows = (db.query(
        TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id,
        TLS211_PAT_PUBLN.publn_auth, TLS201_APPLN.appln_filing_year)
    .join(TLS211_PAT_PUBLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids))
    .distinct().all())
auth = pd.DataFrame(rows, columns=['docdb_family_id', 'appln_id', 'publn_auth', 'filing_year'])
print(f"{len(auth):,} distinct filings across {auth['publn_auth'].nunique()} authorities.")

## Step 3 — Filings by international / regional authority

We focus on the five **supranational** routes — WO (PCT), EP (European Patent Office), and the regional offices EA (Eurasian), AP (ARIPO), OA (OAPI) — because they show the international ambition of the field. This is the first chart of the notebook.

In [ ]:
import plotly.express as px

intl = ["WO", "EP", "EA", "AP", "OA"]
labels = {"WO": "WO — PCT (WIPO)", "EP": "EP — European Patent Office",
          "EA": "EA — Eurasian Patent Org.", "AP": "AP — ARIPO (Africa)",
          "OA": "OA — OAPI (Africa)"}
intl_df = auth[auth['publn_auth'].isin(intl)]
totals = intl_df['publn_auth'].value_counts().reindex(intl).fillna(0).astype(int)
totals_tbl = pd.DataFrame({'authority': [labels[a] for a in totals.index],
                           'filings': totals.values})

fig = px.bar(totals_tbl, x='authority', y='filings', text='filings')
fig.update_traces(marker_color='#4527A0', textposition='outside')
fig.update_layout(title='Filings by international / regional authority',
                  xaxis_title='Authority', yaxis_title='Number of filings',
                  template='plotly_white', height=500)

report_kit.record(210, "intl_authority_totals",
                  "Filings by international / regional authority", fig, totals_tbl, output_dir=OUT,
                  note="Counts distinct applications (filings), not families. WO/EP/EA/AP/OA.")

## Step 4 — WO (PCT) vs EP over time

The two dominant routes — the global PCT filing (WO) and the European route (EP) — plotted per filing year from 2000 to 2023. It shows how the field's international vs European footprint evolved.

In [ ]:
yearly = (intl_df[intl_df['publn_auth'].isin(['WO', 'EP'])]
          .groupby(['filing_year', 'publn_auth']).size().reset_index(name='filings'))
yearly = yearly[(yearly['filing_year'] >= 2000) & (yearly['filing_year'] <= 2023)]

fig = px.line(yearly, x='filing_year', y='filings', color='publn_auth', markers=True,
              labels={'filing_year': 'Filing year', 'filings': 'Number of filings',
                      'publn_auth': 'Authority'})
fig.update_layout(title='WO (PCT) vs EP filings over time',
                  template='plotly_white', height=500)

report_kit.record(220, "intl_authority_trend",
                  "WO (PCT) vs EP over time", fig, yearly, output_dir=OUT,
                  note="Filings per year for WO and EP, 2000–2023.")

## Step 5 — National filing trends

Beyond the supranational routes, where are inventions taken **nationally**? We reuse the filings
already retrieved in Step 2, drop the supranational authorities (WO/EP/EA/AP/OA), and stack the
busiest single-country offices over time — a first read on which national markets matter.

In [ ]:
# reuse `auth` from Step 2; national offices = everything except the supranational routes
national = auth[~auth['publn_auth'].isin(['WO', 'EP', 'EA', 'AP', 'OA'])]
top_nat = national['publn_auth'].value_counts().head(8).index.tolist()
nat_year = (national[national['publn_auth'].isin(top_nat)]
            .groupby(['filing_year', 'publn_auth']).size().reset_index(name='filings'))
nat_year = nat_year[(nat_year['filing_year'] >= 2000) & (nat_year['filing_year'] <= 2023)]

fig = px.area(nat_year, x='filing_year', y='filings', color='publn_auth',
              labels={'filing_year': 'Filing year', 'filings': 'Number of filings', 'publn_auth': 'Office'})
fig.update_layout(title='National filing trends — the busiest single-country offices',
                  template='plotly_white', height=500)

report_kit.record(230, "national_filing_trends",
                  "National filing trends", fig, nat_year, output_dir=OUT,
                  note="Filings per year at the top 8 national offices, 2000–2023 (per application).")

## Step 6 — Innovation waves by technology area

How has *what* the field patents shifted over time? We pull each family's IPC codes with its
earliest filing year, group the IPC subclasses into a few readable technology areas, and stack
families per year — successive "waves" of drugs, diagnostics, microbiology and chemistry.

In [ ]:
from epo.tipdata.patstat.database.models import TLS209_APPLN_IPC

ipc_year = pd.DataFrame(
    db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.earliest_filing_year,
             TLS209_APPLN_IPC.ipc_class_symbol)
      .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
      .filter(TLS201_APPLN.docdb_family_id.in_(family_ids))
      .filter(TLS201_APPLN.earliest_filing_year >= 2000).all(),
    columns=['docdb_family_id', 'year', 'ipc'])

def tech_area(sym):
    sub = str(sym)[:4]
    if sub in ('A61K', 'A61P'):         return 'Drugs & therapeutics'
    if sub in ('C12Q', 'G01N'):         return 'Diagnostics & testing'
    if sub in ('C12N', 'C12P'):         return 'Microbiology & genetics'
    if sub in ('C07D', 'C07K', 'C07C'): return 'Chemistry'
    if sub in ('A61L', 'A01N'):         return 'Materials & biocides'
    return 'Other'

ipc_year['area'] = ipc_year['ipc'].map(tech_area)
# count each family once per area and year it appears in
waves = (ipc_year.drop_duplicates(['docdb_family_id', 'area', 'year'])
         .groupby(['year', 'area']).size().reset_index(name='families'))
waves = waves[(waves['year'] >= 2000) & (waves['year'] <= 2023)]

fig = px.area(waves, x='year', y='families', color='area',
              labels={'year': 'Earliest filing year', 'families': 'Patent families', 'area': 'Technology area'})
fig.update_layout(title='Innovation waves — families over time by technology area',
                  template='plotly_white', height=500)

report_kit.record(240, "innovation_waves",
                  "Innovation waves by technology area", fig, waves, output_dir=OUT,
                  note="Families per year, split by broad technology area (grouped IPC subclasses).")

## Step 7 — National vs international filing strategy

A single ratio tells the internationalisation story: of all applications filed each year, what
share goes through an **international / regional** route (EP, WO, OA, AP, EA) versus a purely
**national** one? We count distinct applications per year and stack the two types.

In [ ]:
apps = pd.DataFrame(
    db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id,
             TLS201_APPLN.appln_auth, TLS201_APPLN.earliest_filing_year)
      .filter(TLS201_APPLN.docdb_family_id.in_(family_ids)).distinct().all(),
    columns=['docdb_family_id', 'appln_id', 'appln_auth', 'year'])
apps = apps[(apps['year'] >= 2000) & (apps['year'] <= 2023)]

intl_auth = ['EP', 'WO', 'OA', 'AP', 'EA']
apps['type'] = apps['appln_auth'].apply(lambda a: 'International' if a in intl_auth else 'National')
by_type = (apps.groupby(['year', 'type'])['appln_id'].nunique()
           .reset_index(name='applications'))

fig = px.area(by_type, x='year', y='applications', color='type',
              color_discrete_map={'National': '#94a3b8', 'International': '#be0f05'},
              labels={'year': 'Earliest filing year', 'applications': 'Applications', 'type': 'Route'})
fig.update_layout(title='National vs international filing strategy over time',
                  template='plotly_white', height=500)

report_kit.record(250, "national_vs_international",
                  "National vs international filing strategy", fig, by_type, output_dir=OUT,
                  note="Distinct applications per year by route; international = EP/WO/OA/AP/EA.")

## Step 8 — Family size & global reach

How big are these patent families? PATSTAT stores `docdb_family_size` (the number of applications
in a family) — a proxy for how widely an invention is pursued. We take one value per family and
show the distribution across size bands.

In [ ]:
import numpy as np

sizes = pd.DataFrame(
    db.query(TLS201_APPLN.docdb_family_id, TLS201_APPLN.docdb_family_size)
      .filter(TLS201_APPLN.docdb_family_id.in_(family_ids)).distinct().all(),
    columns=['docdb_family_id', 'family_size']).drop_duplicates('docdb_family_id')

bands = ['1', '2–3', '4–6', '7–10', '11–20', '21+']
sizes['band'] = pd.cut(sizes['family_size'], bins=[0, 1, 3, 6, 10, 20, np.inf], labels=bands)
dist = (sizes['band'].value_counts().reindex(bands).fillna(0).astype(int)
        .rename_axis('family_size').reset_index(name='families'))

fig = px.bar(dist, x='family_size', y='families', text='families')
fig.update_traces(marker_color='#1f6feb', textposition='outside')
fig.update_layout(title='Family size & global reach — distribution of DOCDB family size',
                  xaxis_title='Applications per family', yaxis_title='Patent families',
                  template='plotly_white', height=480)

report_kit.record(255, "family_size",
                  "Family size & global reach", fig, dist, output_dir=OUT,
                  note="One docdb_family_size per family, grouped into size bands.")

## Step 9 — Top applicants by patent families

Who files the most? We attach applicants (persons with `applt_seq_nr != 0`) to the families and
count **distinct families per applicant** — a clean volume leaderboard of the field's most active
organisations.

In [ ]:
from epo.tipdata.patstat.database.models import TLS206_PERSON, TLS207_PERS_APPLN

appl = pd.DataFrame(
    db.query(TLS206_PERSON.psn_name, TLS201_APPLN.docdb_family_id)
      .join(TLS207_PERS_APPLN, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
      .join(TLS201_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
      .filter(TLS207_PERS_APPLN.applt_seq_nr != 0,
              TLS201_APPLN.docdb_family_id.in_(family_ids),
              TLS201_APPLN.earliest_filing_year.between(2000, 2023)).all(),
    columns=['applicant', 'docdb_family_id'])

rank = (appl.groupby('applicant')['docdb_family_id'].nunique()
        .reset_index(name='families').sort_values('families', ascending=False))
top = rank.head(15)

fig = px.bar(top.sort_values('families'), x='families', y='applicant', orientation='h', text='families')
fig.update_traces(marker_color='#be0f05', textposition='outside')
fig.update_layout(title='Top 15 applicants by patent families',
                  xaxis_title='Distinct patent families', yaxis_title='',
                  template='plotly_white', height=560)

report_kit.record(260, "top_applicants",
                  "Top applicants by patent families", fig, top, output_dir=OUT,
                  note="Distinct families per applicant (psn_name), applicants only, 2000–2023.")

## Step 10 — Applicants by institutional sector

PATSTAT classifies each person by `psn_sector`. We map it to readable categories (company,
university, hospital, government / non-profit, individual) and count families per sector — the
institutional make-up of who drives the field.

In [ ]:
sec = pd.DataFrame(
    db.query(TLS206_PERSON.psn_sector, TLS201_APPLN.docdb_family_id)
      .join(TLS207_PERS_APPLN, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
      .join(TLS201_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
      .filter(TLS207_PERS_APPLN.applt_seq_nr != 0,
              TLS201_APPLN.docdb_family_id.in_(family_ids),
              TLS201_APPLN.earliest_filing_year.between(2000, 2023)).all(),
    columns=['psn_sector', 'docdb_family_id'])

def sector_label(s):
    s = str(s).strip().upper() if pd.notna(s) else ''
    if 'UNIVERSITY' in s:                    return 'University'
    if 'HOSPITAL' in s:                      return 'Hospital'
    if 'GOV' in s or 'NON-PROFIT' in s:      return 'Government / Non-profit'
    if 'COMPANY' in s:                       return 'Company'
    if 'INDIVIDUAL' in s or s == 'IND':      return 'Individual'
    return 'Other / Unknown'

sec['sector'] = sec['psn_sector'].map(sector_label)
sector_dist = (sec.drop_duplicates(['docdb_family_id', 'sector'])
               .groupby('sector')['docdb_family_id'].nunique()
               .reset_index(name='families').sort_values('families', ascending=False))

fig = px.pie(sector_dist, names='sector', values='families', hole=0.45)
fig.update_traces(textinfo='label+percent')
fig.update_layout(title='Applicants by institutional sector', template='plotly_white', height=520)

report_kit.record(270, "applicants_by_sector",
                  "Applicants by institutional sector", fig, sector_dist, output_dir=OUT,
                  note="Families per applicant sector (psn_sector), counted once per family-sector.")

## Step 11 — Grant rate by top applicants

How often do the leading applicants actually get their applications **granted**? PATSTAT's
`granted` flag on the application says whether it issued. We compute granted / total per applicant
for those with enough volume to be meaningful (≥ 10 applications).

In [ ]:
gr = pd.DataFrame(
    db.query(TLS201_APPLN.appln_id, TLS201_APPLN.granted, TLS206_PERSON.psn_name)
      .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
      .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
      .filter(TLS201_APPLN.docdb_family_id.in_(family_ids),
              TLS201_APPLN.earliest_filing_year.between(2000, 2023),
              TLS207_PERS_APPLN.applt_seq_nr != 0).all(),
    columns=['appln_id', 'granted', 'applicant'])

agg = (gr.groupby('applicant')
       .agg(total=('appln_id', 'size'), granted=('granted', lambda s: (s == 'Y').sum()))
       .reset_index())
agg['grant_rate'] = (agg['granted'] / agg['total'] * 100).round(1)
top_vol = agg[agg['total'] >= 10].sort_values('total', ascending=False).head(20)

fig = px.bar(top_vol, x='applicant', y='grant_rate', text='grant_rate')
fig.update_traces(marker_color='#2e8540', texttemplate='%{text}%', textposition='outside')
fig.update_layout(title='Grant rate — top 20 applicants by application volume',
                  xaxis_title='', yaxis_title='Grant rate (%)', xaxis_tickangle=-45,
                  template='plotly_white', height=560)

report_kit.record(280, "grant_rate",
                  "Grant rate by top applicants", fig, top_vol, output_dir=OUT,
                  note="Granted (TLS201.granted='Y') / total applications per applicant; ≥10 applications.")

## Step 12 — Most influential organisations (forward citations)

Finally, whose antibiotic-resistance patents are **cited most** by later patents? Using BigQuery's
DOCDB family-citation table, we count how many distinct later families cite each organisation's
patents in our set — a measure of influence. *(This is the clean version: we count distinct citing
families per organisation, and keep organisational applicants only.)*

In [ ]:
# BigQuery (citation self-joins are the honest tool here) — fully-qualified PATSTAT tables
fam_str = ", ".join(str(int(f)) for f in family_ids)
sql = f"""
SELECT c.docdb_family_id AS citing_family_id, p.psn_name AS applicant
FROM `p-epo-tip-prj-3a1f.p_epo_tip_euwe4_bqd_patstatb.tls228_docdb_fam_citn` AS c
JOIN `p-epo-tip-prj-3a1f.p_epo_tip_euwe4_bqd_patstatb.tls201_appln` AS a
  ON c.cited_docdb_family_id = a.docdb_family_id
JOIN `p-epo-tip-prj-3a1f.p_epo_tip_euwe4_bqd_patstatb.tls207_pers_appln` AS pa
  ON pa.appln_id = a.appln_id
JOIN `p-epo-tip-prj-3a1f.p_epo_tip_euwe4_bqd_patstatb.tls206_person` AS p
  ON p.person_id = pa.person_id
WHERE c.cited_docdb_family_id IN ({fam_str})
  AND pa.applt_seq_nr > 0
  AND p.psn_name IS NOT NULL
  AND p.psn_sector IN ('COMPANY', 'UNIVERSITY', 'GOV NON-PROFIT', 'COMPANY GOV NON-PROFIT')
"""
cit = pd.DataFrame(patstat.sql_query(sql, use_legacy_sql=False))

rank = (cit.groupby('applicant')['citing_family_id'].nunique()
        .reset_index(name='citing_families').sort_values('citing_families', ascending=False))
top = rank.head(15)

fig = px.bar(top.sort_values('citing_families'), x='citing_families', y='applicant',
             orientation='h', text='citing_families')
fig.update_traces(marker_color='#6f42c1', textposition='outside')
fig.update_layout(title='Most influential organisations — distinct citing families',
                  xaxis_title='Distinct later families citing them', yaxis_title='',
                  template='plotly_white', height=560)

report_kit.record(290, "top_cited_orgs",
                  "Most influential organisations (forward citations)", fig, top, output_dir=OUT,
                  note="Distinct citing families per cited organisation; DOCDB family citations; organisational applicants only.")

---
**Done.** The core-landscape battery is recorded — ten charts from `dataset.xlsx`, covering where
the field files, how it internationalises, how big its families are, and who leads it. Continue
with notebook 3 (advanced analyses), then notebook 4 assembles the report.